©2026. For information, contact Deloitte Tohmatsu Group.

# 📝 演習概要

この演習では、ニューラルネットワーク（深層学習モデル）に対するハイパーパラメータ調整を実践します。scikit-learnのMLPClassifierにGridSearchとRandomSearchを適用し、隠れ層サイズや学習率などのパラメータ最適化を行い、各探索手法の結果を比較します。

# 【3-2】ハイパーパラメータの調整

ここでは、乳がんのデータセットを用いて、**ハイパーパラメータの調整**について理解します．


**【目標】<font color="red">ハイパーパラメータの調整手法を実装できる．</font>**

## ０．各ライブラリを読み込む

In [ ]:
# データ加工・処理・分析モジュール
import pandas as pd
import numpy as np

## １．データを読み込む

In [ ]:
# scikit-learnから乳がんデータセットを取得する
from sklearn import datasets
data = datasets.load_breast_cancer()

データを pandas ライブラリの DataFrame形式に変換します.

In [ ]:
# 特徴量をpandasのDataFrameに詰めることで、列名（=特徴量名）付きの見やすい表にする
data_df = pd.DataFrame(data.data, columns=data.feature_names)

# 先頭5行を確認
data_df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## ２．ニューラルネットワークに"グリッドサーチ"と"ランダムサーチ"を用いて予測する
ここでは、`MLPClassifier`（多層パーセプトロン）を用いて、ニューラルネットワークのハイパーパラメータを調整します.
**グリッドサーチ**と**ランダムサーチ**の両方を試し、探索方法の違いによる結果の違いを比較します.

In [ ]:
# 学習用と評価用にデータを分割
from sklearn.model_selection import train_test_split

# - X_train, y_train: 学習用データ
# - X_test,  y_test : 最終評価用データ
# test_size=0.2 は 20% をテスト用にするという意味
# random_state=42 で分割を再現可能にしている
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

scikit-learn ライブラリの GridSearchCV メソッドを用いることでグリッドサーチを行うことができます．

In [ ]:
# ニューラルネットワーク（MLPClassifier）とグリッドサーチ
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV

# 探索したいハイパーパラメータの候補を定義
# - hidden_layer_sizes:
#     (50,)     -> 隠れ層1層、ユニット数50
#     (100,)    -> 隠れ層1層、ユニット数100
#     (50, 50)  -> 隠れ層2層、各層50ユニット
#
# - learning_rate_init: 学習率の候補
param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'learning_rate_init': [0.001, 0.01, 0.1]
}

# MLPClassifier（全結合ニューラルネット）
# - max_iter=1000: 最大エポック数
# - random_state=0: 重み初期化などの乱数の固定
mlp = MLPClassifier(max_iter=1000, random_state=0)

# - param_grid内のすべての組み合わせを総当たりで試す
# - cv=5: 5分割交差検証でスコアを評価
clf = GridSearchCV(mlp, param_grid, cv=5)

# グリッドサーチ付きで学習
clf.fit(X_train, y_train)

print("[グリッドサーチ] 最適なパラメータ:", clf.best_params_)
print("[グリッドサーチ] テストデータ正解率:", clf.score(X_test, y_test))

[グリッドサーチ] 最適なパラメータ: {'hidden_layer_sizes': (100,), 'learning_rate_init': 0.001}
[グリッドサーチ] テストデータ正解率: 0.9298245614035088


※ cv という値を設定することで、より細かいクロスバリデーションを設定できるが、どんどん処理時間が長くなります．

## ３．ランダムサーチによるハイパーパラメータの調整
`RandomizedSearchCV` を使ってランダムサーチを実施します.探索範囲はグリッドサーチと似ていますが、より広く柔軟に試すことができます.

In [ ]:
# ランダムサーチの実装
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

# ランダムサーチ用の分布・候補を定義
param_dist = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)],
    'learning_rate_init': uniform(0.0001, 0.1)
}

# 別のMLPインスタンスを用意
mlp2 = MLPClassifier(max_iter=1000, random_state=1)

# - n_iter=10: 10パターンだけ試す
# - cv=5: 5分割交差検証
# - random_state=42: 乱数固定で再現性を確保
random_search = RandomizedSearchCV(mlp2, param_distributions=param_dist, n_iter=10, cv=5, random_state=42)
random_search.fit(X_train, y_train)


# 最良パラメータとテストセット精度を表示
print("[ランダムサーチ] 最適なパラメータ:", random_search.best_params_)
print("[ランダムサーチ] テストデータ正解率:", random_search.score(X_test, y_test))

[ランダムサーチ] 最適なパラメータ: {'hidden_layer_sizes': (100,), 'learning_rate_init': np.float64(0.018282496720710063)}
[ランダムサーチ] テストデータ正解率: 0.9649122807017544


## 🔧 実践問題1：活性化関数とsolverも探索対象に加えてグリッドサーチを拡張する

上のコードでは `hidden_layer_sizes` と `learning_rate_init` だけを探索しましたが、
MLPClassifierには他にも精度に大きく影響するハイパーパラメータがあります。

| パラメータ | 選択肢 | 意味 |
|:---|:---|:---|
| `activation` | `'relu'`, `'tanh'`, `'logistic'` | 隠れ層の活性化関数 |
| `solver` | `'adam'`, `'sgd'`, `'lbfgs'` | 最適化手法。adamが最も一般的 |
| `alpha` | `0.0001`, `0.001`, `0.01` | L2正則化の強さ。過学習防止 |

ただし、探索パラメータを増やすと組み合わせ数が爆発します。

---

**問題：** 以下のコードの `______` を埋めて、活性化関数とsolverも含めたグリッドサーチを実行し、探索の組み合わせ数と計算時間の変化を確認してください。

<br/>

<details>
<summary>💡 <b>ヒント（クリックして表示）</b></summary>

<blockquote>
組み合わせ数 = 各パラメータの候補数の積 です。<code>cv=3</code> にすると <code>cv=5</code> より速く結果が出ます。
</blockquote>

</details>

<br/>


In [ ]:
import time
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV

# 探索範囲を拡張
param_grid2 = {
    'hidden_layer_sizes': [(100,), (50, 50)],
    'learning_rate_init': [0.001, 0.01],
    '______': ['relu', '______'],    # 活性化関数
    '______': ['adam', '______'],     # 最適化手法
}

# 組み合わせ数を計算して表示
n_combinations = 1
for key, values in param_grid2.items():
    n_combinations ______ len(values)
print(f'探索する組み合わせ数: {n_combinations}')

mlp3 = MLPClassifier(max_iter=1000, random_state=0)
clf3 = GridSearchCV(mlp3, param_grid2, cv=3, verbose=1)

start = time.time()
clf3.fit(X_train, y_train)
elapsed = time.time() - start

print(f'\n探索時間: {elapsed:.1f}秒')
print(f'最適パラメータ: {clf3.best_params_}')
print(f'テスト精度: {clf3.score(X_test, y_test):.3f}')

# 上のグリッドサーチ（9通り）の結果と比較
print(f'\n※ 上のグリッドサーチ結果: {clf.best_params_}')
print(f'※ 上のテスト精度: {clf.score(X_test, y_test):.3f}')


<details><summary>解答例</summary>

```python
param_grid2 = {
    'hidden_layer_sizes': [(100,), (50, 50)],
    'learning_rate_init': [0.001, 0.01],
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'sgd'],
}

n_combinations *= len(values)  # *= で掛け算して累積
```

- `activation` と `solver` を追加すると 2×2×2×2 = **16通り** になります。元の9通りと比べて増えましたが、各パラメータの候補を2つに絞ることで現実的な探索時間に収まります
- `'tanh'` はシグモイドに似た形状で出力が-1〜1。`'relu'` はディープラーニングで最も一般的な活性化関数です
- `'sgd'` は素朴な確率的勾配降下法で、`'adam'` より学習率の調整がシビアですが、うまくいくと汎化性能が良い場合があります
- `n_combinations *= len(values)` で候補数の積を計算しています。パラメータが増えるほど組み合わせは指数的に増えるため、グリッドサーチが現実的でなくなることがあります。そのような場合にランダムサーチが有効です
</details>


## 🔧 実践問題2：前処理（標準化）の有無が探索結果に与える影響を確認する

上のコードではデータをそのまま学習に使っていますが、ニューラルネットワークは**入力の特徴量のスケールが揃っていないと学習がうまくいかない**ことがあります。

乳がんデータセットの特徴量は、値の範囲がバラバラです：
- `mean radius`: 6〜28 程度
- `mean area`: 143〜2501 程度
- `mean smoothness`: 0.05〜0.16 程度

`StandardScaler` で**標準化（平均0、分散1）**すると、全特徴量が同じスケールに揃い、学習が安定します。

scikit-learnの `Pipeline` を使うと、前処理とモデルをまとめてグリッドサーチに渡せます。

---

**問題：** 以下のコードの `______` を埋めて、標準化→MLPの `Pipeline` を作成し、標準化なしの結果と精度を比較してください。

<br/>

<details>
<summary>💡 <b>ヒント（クリックして表示）</b></summary>

<blockquote>
<code>Pipeline</code> 内のパラメータ名は <code>'ステップ名__パラメータ名'</code> の形式です（アンダースコア2つ）。
</blockquote>

</details>

<br/>


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import ______  # 標準化を行うクラス
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV

# Pipeline: 標準化 → MLP の2ステップ
pipe = Pipeline([
    ('scaler', ______()),     # ステップ名 'scaler' で標準化
    ('mlp', MLPClassifier(max_iter=1000, random_state=0))
])

# Pipeline内のMLPのパラメータを探索するには 'mlp__パラメータ名' と書く
param_grid_pipe = {
    'mlp__hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'mlp________': [0.001, 0.01, 0.1],  # 学習率
}

clf_pipe = GridSearchCV(pipe, param_grid_pipe, cv=5)
clf_pipe.______(X_train, y_train)

print('--- 標準化あり（Pipeline） ---')
print(f'最適パラメータ: {clf_pipe.best_params_}')
print(f'テスト精度: {clf_pipe.score(X_test, y_test):.3f}')

print('\n--- 標準化なし（上のグリッドサーチ） ---')
print(f'最適パラメータ: {clf.best_params_}')
print(f'テスト精度: {clf.score(X_test, y_test):.3f}')


<details><summary>解答例</summary>

```python
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(max_iter=1000, random_state=0))
])

param_grid_pipe = {
    'mlp__hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'mlp__learning_rate_init': [0.001, 0.01, 0.1],
}

clf_pipe = GridSearchCV(pipe, param_grid_pipe, cv=5)
clf_pipe.fit(X_train, y_train)
```

- `StandardScaler` は各特徴量を平均0・分散1に標準化します。`from sklearn.preprocessing import StandardScaler` でインポートします
- `Pipeline` で前処理とモデルをまとめると、交差検証の各foldで**学習データだけを使ってスケーラーを学習**し、テストデータには同じ変換を適用します。これにより**データリーク**を防げます
- Pipeline内のパラメータ名は `'ステップ名__パラメータ名'`（アンダースコア2つ）です。`'mlp__learning_rate_init'` のように書きます
- 標準化によって精度が大きく改善するはずです。ニューラルネットワークは入力スケールに敏感なため、標準化は**ほぼ必須の前処理**です
- SVMでも `StandardScaler` は有効です。ML版の演習でも試してみてください
</details>
